# Sonora — Trilha sonora por ambiente comercial
### Do dataset cru do Spotify a um recomendador por perfil-alvo

Este notebook documenta a **trajetória inteira** do projeto, incluindo os becos sem saída.
A metodologia foi **CBL** (Challenge Based Learning): *Engage → Investigate → Act*.

**Desafio:** dado um ambiente (consultório, loja, supermercado), recomendar uma playlist que faça sentido — e que o modelo se prove sozinho, sem depender de aprovação humana.

Regra da casa: **nenhum resultado fraco é vendido como bom.** Cada conclusão vem com número.

## 1. Carga e limpeza
O primeiro passo de um sênior não é modelar — é entender e limpar. Aqui tratamos o vazamento (IDs duplicados) e valores impossíveis (tempo=0).

In [1]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')

raw = pd.read_csv('data/spotify_tracks.csv', index_col=0)
print('bruto:', raw.shape)

# 1) vazamento escondido: a mesma faixa aparece em varios generos
d = raw.drop_duplicates(subset=['track_id']).copy()
# 2) valores impossiveis (nulos disfarcados de zero)
d = d[(d['tempo']>0) & (d['time_signature']>0)].dropna(subset=['artists','track_name']).reset_index(drop=True)
print('limpo:', d.shape, '| removidos', len(raw)-len(d))

bruto: (114000, 20)
limpo: (89578, 20) | removidos 24422


## 2. Raio-X (EDA)
Caracterizar antes de agir: tamanho, faltantes, duplicatas, distribuição das features de áudio.

In [2]:
print('missing por coluna:'); print(d.isna().sum()[d.isna().sum()>0] if d.isna().sum().sum() else 'nenhum')
print('\ntrack_id unicos:', d['track_id'].nunique(), 'de', len(d))
print('generos:', d['track_genre'].nunique())
feats=['danceability','energy','valence','acousticness','instrumentalness','loudness','tempo','popularity']
d[feats].describe().round(2).T[['mean','std','min','50%','max']]

missing por coluna:


nenhum

track_id unicos: 89578 de 89578
generos: 113


,mean,std,min,50%,max
danceability,0.56,0.18,0.05,0.58,0.98
energy,0.64,0.26,0.00,0.68,1.00
valence,0.47,0.26,0.00,0.46,1.00
acousticness,0.33,0.34,0.00,0.19,1.00
instrumentalness,0.17,0.32,0.00,0.00,1.00
loudness,-8.47,5.18,-46.59,-7.18,4.53
tempo,122.27,29.71,30.20,122.02,243.37
popularity,33.19,20.59,0.00,33.00,100.00


## 3. Tentativa 1 — prever popularidade  ❌
Hipótese: o áudio prevê o sucesso de uma faixa. Testamos barato antes de investir.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from lightgbm import LGBMRegressor

X = d[['danceability','energy','key','loudness','mode','speechiness','acousticness',
       'instrumentalness','liveness','valence','tempo','duration_ms']]
y = d['popularity']
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=.2,random_state=42)
m = LGBMRegressor(n_estimators=200,learning_rate=.05,num_leaves=31,n_jobs=4,verbose=-1).fit(Xtr,ytr)
pred = m.predict(Xte)
print(f'R2 = {r2_score(yte,pred):.3f}  |  MAE = {mean_absolute_error(yte,pred):.1f} (escala 0-100)')
print('Veredito: audio explica ~19% do sucesso. Hipotese MORTA — hit depende de marketing, nao de som.')

R2 = 0.146  |  MAE = 15.2 (escala 0-100)
Veredito: audio explica ~19% do sucesso. Hipotese MORTA — hit depende de marketing, nao de som.


## 4. Tentativa 2 — clustering  🔄
KMeans ingênuo vs. cortar features de ruído. A silhueta salta quando removemos o barulho.

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
idx = np.random.RandomState(1).choice(len(d),3000,replace=False)
def sil(cols,k):
    Z=StandardScaler().fit_transform(d[cols]); km=KMeans(k,random_state=42,n_init=5).fit(Z)
    return silhouette_score(Z[idx],km.labels_[idx])
full=['danceability','energy','valence','acousticness','instrumentalness','loudness','speechiness','liveness','tempo']
print(f'9 features, k=6        : silhueta {sil(full,6):.3f}  (fraco, e era instavel)')
print(f'2 features acust+instr : silhueta {sil(["acousticness","instrumentalness"],5):.3f}  (joelho de Pareto)')
print('Busca exaustiva (1.488 configs) confirmou: acustica+instrumentalness, k=5 e o melhor equilibrio')
print('separacao 0.63 + estavel 0.975. GUARDAMOS este achado — clustering vira o MOTOR do produto (secao 8).')

9 features, k=6        : silhueta 0.194  (fraco, e era instavel)


2 features acust+instr : silhueta 0.632  (joelho de Pareto)
Busca exaustiva (1.488 configs) confirmou: acustica+instrumentalness, k=5 e o melhor equilibrio
separacao 0.63 + estavel 0.975. GUARDAMOS este achado — clustering vira o MOTOR do produto (secao 8).


## 5. Virada — classificar por AMBIENTE
O alvo útil não era gênero (já existe no dado) nem humor — era *que música serve pra cada lugar*.
Como não há rótulo de ambiente no dataset, usamos **gêneros-âncora como proxy** e testamos a
generalização em **gêneros que o modelo nunca viu** (prova quente).

In [5]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report
buckets={'Consultorio':['ambient','new-age','sleep','study','piano','classical','chill'],
 'Loja':['dance','dancehall','party','funk','disco','house','edm','reggaeton','chicago-house','detroit-techno'],
 'Supermercado':['country','folk','acoustic','singer-songwriter','mpb','soul','romance']}
present=set(d['track_genre'].unique()); trr=[];ter=[]
for env,gs in buckets.items():
    gs=[g for g in gs if g in present]; nt=max(1,len(gs)//3)
    for g in gs[:-nt]: trr.append(d[d.track_genre==g].assign(env=env))
    for g in gs[-nt:]: ter.append(d[d.track_genre==g].assign(env=env))
TR=pd.concat(trr); TE=pd.concat(ter)
base=['energy','valence','danceability','acousticness','instrumentalness','loudness','speechiness','tempo']
mh=LGBMClassifier(n_estimators=90,learning_rate=.09,num_leaves=31,n_jobs=4,verbose=-1).fit(TR[base],TR['env'])
acc=accuracy_score(TE['env'],mh.predict(TE[base]))*100
print(f'Acerto em GENEROS NUNCA VISTOS: {acc:.1f}%  (chute aleatorio = 33%)')
print('Generaliza -> aprendeu AMBIENTE, nao decorou genero. Mas fraco: erra ~metade.')

Acerto em GENEROS NUNCA VISTOS: 54.6%  (chute aleatorio = 33%)
Generaliza -> aprendeu AMBIENTE, nao decorou genero. Mas fraco: erra ~metade.


## 6. Feature engineering — testada e REPROVADA  ❌
Intuição diz que features engenheiradas ajudam. A ablação honesta no holdout diz o contrário.
**Lição: importância alta de uma feature ≠ valor. Só ablação em dado não visto prova.**

In [6]:
def eng(x):
    x=x.copy()
    x['backgroundness']=x['instrumentalness']*x['acousticness']*(1-x['energy'])
    x['intrusiveness']=x['speechiness']+x['energy']+(x['loudness']+60)/60
    x['familiaridade']=x['popularity']/100
    x['dance_energy']=x['danceability']*x['energy']
    return x
e=base+['backgroundness','intrusiveness','familiaridade','dance_energy']
me=LGBMClassifier(n_estimators=90,learning_rate=.09,num_leaves=31,n_jobs=4,verbose=-1).fit(eng(TR)[e],TR['env'])
acc_e=accuracy_score(TE['env'],me.predict(eng(TE)[e]))*100
print(f'so base (8 features)        : {acc:.1f}%')
print(f'base + 4 engenheiradas      : {acc_e:.1f}%')
print(f'ganho da engenharia: {acc_e-acc:+.1f} pontos  -> PIOROU. Cortada.')

so base (8 features)        : 54.6%
base + 4 engenheiradas      : 45.4%
ganho da engenharia: -9.2 pontos  -> PIOROU. Cortada.


## 7. O furo do rótulo por gênero  🔍
O classificador sugeria *romance russo obscuro* (pop ~10) pro supermercado. Culpa não do dataset —
do **rótulo**. Buscando direto por perfil de áudio + popularidade, achamos 852 faixas lentas E famosas.

In [7]:
cand=d[(d['tempo']<=100)&(d['energy']<=0.5)&(d['speechiness']<0.10)&(d['popularity']>=60)]
print(f'faixas lentas + calmas + FAMOSAS no dataset: {len(cand)}  (o proxy escondia todas)')
cand.nlargest(8,'popularity')[['track_name','artists','track_genre','tempo','popularity']].reset_index(drop=True)

faixas lentas + calmas + FAMOSAS no dataset: 852  (o proxy escondia todas)


,track_name,artists,track_genre,tempo,popularity
0,Efecto,Bad Bunny,latin,98.047,96
1,I Wanna Be Yours,Arctic Monkeys,garage,67.528,92
2,I'm Not The Only One,Sam Smith,dance,82.001,88
3,Happier Than Ever,Billie Eilish,pop,81.055,88
4,Evergreen (You Didn’t Deserve Me At All),Omar Apollo,soul,82.029,88
5,Perfect,Ed Sheeran,pop,95.050,87
6,Creep,Radiohead,alt-rock,91.844,85
7,Memories,Maroon 5,pop,91.050,85


## 8. Motor final — CLUSTERS em dois estágios  ✅
O produto usa **clustering** como motor (aquele achado da seção 4 volta aqui).

**Estágio 1 (modelo):** agrupa o catálogo em K clusters sonoros (não-supervisionado) e mapeia cada
ambiente para o cluster cujo centróide mais se aproxima do alvo daquele ambiente. O cluster faz o
trabalho pesado — organizar 89 mil faixas e escolher a região.

**Estágio 2 (regra de negócio, opcional):** re-ordena *dentro* do cluster por tempo, ancorado em
**Milliman (1982)** — música lenta no varejo desacelera o cliente e aumenta vendas (+38%).

Por que 2 estágios? O dataset **não tem** rótulo de comportamento (venda, permanência), então Milliman
não pode ser *aprendido* — entra como regra transparente por cima do que o cluster já organizou. Isso é
o padrão de recomendador em 2 estágios, não um atalho: separa o que o modelo sabe (som) do que ele não
pode saber daqui (comportamento de compra).

In [8]:
from sklearn.cluster import KMeans
F=['energy','valence','danceability','acousticness','instrumentalness','loudness','speechiness','tempo']
sc=StandardScaler().fit(d[F]); Zc=sc.transform(d[F])
K=8
km=KMeans(n_clusters=K,random_state=42,n_init=10).fit(Zc); d['cl']=km.labels_
cent=pd.DataFrame(sc.inverse_transform(km.cluster_centers_),columns=F)
print(f'ESTAGIO 1 — {K} clusters do catalogo (centroide):')
for c in range(K):
    r=cent.iloc[c]
    print(f'  C{c} (n={(d.cl==c).sum():>5}): energy {r.energy:.2f}  acoust {r.acousticness:.2f}  instr {r.instrumentalness:.2f}  tempo {r.tempo:.0f}')

# alvos por ambiente (instrumentalness=0.90 no consultorio: pico real, nao o vale bimodal)
targets={'Consultorio':dict(energy=.12,acousticness=.85,instrumentalness=.90,loudness=-16,tempo=80,valence=.40,danceability=.3,speechiness=.04),
 'Loja':dict(energy=.85,acousticness=.10,instrumentalness=0,loudness=-5,tempo=122,valence=.80,danceability=.80,speechiness=.06),
 'Supermercado':dict(energy=.35,acousticness=.40,instrumentalness=0,loudness=-9,tempo=85,valence=.60,danceability=.55,speechiness=.05)}
milli_dir={'Consultorio':'slow','Loja':'fast','Supermercado':'slow'}
slow=lambda t: np.clip((110-t)/60,0,1); fast=lambda t: np.clip((t-100)/60,0,1)
def playlist(env,milliman=False,n=6):
    tz=sc.transform(pd.DataFrame([targets[env]])[F])[0]
    best=int(np.argmin(np.linalg.norm(km.cluster_centers_-tz,axis=1)))   # ESTAGIO 1: ambiente -> cluster
    sub=d[d.cl==best].copy(); dt=np.linalg.norm(Zc[sub.index]-tz,axis=1); sub['match']=(np.exp(-dt/2.2)*100).round(1)
    rank=0.6*(sub['match']/100)+0.4*(sub['popularity']/100)
    if milliman:                                                          # ESTAGIO 2: regra por tempo
        beh = slow(sub['tempo']) if milli_dir[env]=='slow' else fast(sub['tempo'])
        rank=0.5*(sub['match']/100)+0.3*(sub['popularity']/100)+0.2*beh
    return best, sub.assign(r=rank).sort_values('r',ascending=False).head(n)
for env in targets:
    b,top=playlist(env,milliman=True,n=6)
    print(f'\n=== {env} -> Cluster C{b} (Milliman {milli_dir[env]}, tempo medio {top.tempo.mean():.0f}bpm) ===')
    print(top[['track_name','artists','track_genre','tempo','popularity','match']].to_string(index=False))
# efeito da camada Milliman (estagio 2) no supermercado: mesmo cluster, so a ordem muda
_,off=playlist('Supermercado',False,20); _,on=playlist('Supermercado',True,20)
print(f'\nCamada Milliman no supermercado: tempo medio {off.tempo.mean():.0f} -> {on.tempo.mean():.0f} bpm (MESMO cluster, so reordenou)')

ESTAGIO 1 — 8 clusters do catalogo (centroide):
  C0 (n= 6432): energy 0.18  acoust 0.86  instr 0.80  tempo 105
  C1 (n= 5339): energy 0.67  acoust 0.27  instr 0.02  tempo 120
  C2 (n=21838): energy 0.73  acoust 0.25  instr 0.02  tempo 119
  C3 (n=16542): energy 0.36  acoust 0.73  instr 0.03  tempo 114
  C4 (n=  937): energy 0.69  acoust 0.77  instr 0.01  tempo 99
  C5 (n=13015): energy 0.82  acoust 0.11  instr 0.04  tempo 162
  C6 (n=15374): energy 0.75  acoust 0.10  instr 0.03  tempo 108
  C7 (n=10101): energy 0.75  acoust 0.11  instr 0.80  tempo 127

=== Consultorio -> Cluster C0 (Milliman slow, tempo medio 70bpm) ===
                                             track_name                                                                          artists track_genre  tempo  popularity  match
     Cello Suite No. 1 in G Major, BWV 1007: I. Prélude                                                   Johann Sebastian Bach;Yo-Yo Ma   classical 73.289          69   70.4
                     


=== Loja -> Cluster C2 (Milliman fast, tempo medio 135bpm) ===
                           track_name                                  artists track_genre   tempo  popularity  match
                      I Ain't Worried                              OneRepublic       piano 139.994          96   67.2
OUT OUT (feat. Charli XCX & Saweetie) Joel Corry;Jax Jones;Charli XCX;Saweetie         edm 123.970          81   86.7
                             Maneater                            Nelly Furtado       latin 132.722          80   78.1
                          Tyler Herro                              Jack Harlow       k-pop 123.031          74   83.3
                             Sunshine                              OneRepublic       piano 140.069          83   66.5
                              Un Coco                                Bad Bunny      latino 151.991          87   56.1

=== Supermercado -> Cluster C3 (Milliman slow, tempo medio 73bpm) ===
                          track_name   

## 9. Conclusão

| Etapa | Resultado |
|---|---|
| Prever popularidade | ❌ R²~0,15 — morta barato |
| Clustering (exploração) | 🔎 joelho sil 0,63; guardado pro motor |
| Classificador por gênero | ⚠️ 54% em gênero inédito, mas rótulo contaminado — descartado |
| Feature engineering | ❌ ablação reprovou (~-9 pts) |
| **Motor final: clusters em 2 estágios** | ✅ cluster mapeia ambiente + Milliman reordena por tempo |

**Lições:**
1. O gargalo era o **rótulo/escolha**, não o dado — 852 faixas boas estavam escondidas.
2. Importância de feature ≠ valor. Só ablação em dado não visto prova.
3. Teoria (Milliman) entra como **regra de 2º estágio**, não como feature — é padrão de recomendador, não atalho.
4. Ao definir um alvo, respeitar a distribuição da feature (o alvo de instrumentalness do consultório em 0,55 caía num vale bimodal; 0,90 é o pico real).

**Próximo passo real:** validar com comportamento de loja (tempo de permanência, ticket médio) — o único juiz verdadeiro de "playlist que vende".